# Assigment 3

In [1]:
import igl
import numpy as np
import pyvista as pv
import scipy.sparse as sp
pv.set_jupyter_backend('trame')

In [2]:
def to_pyvista_mesh(V, F):
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

def plot_mesh_with_normals(V, F, N, factor=0.1):
    arrows = pv.vector_poly_data(V, N)
    arrows = arrows.glyph(orient='vectors',scale='mag',factor=factor)

    p = pv.Plotter()
    p.add_mesh(to_pyvista_mesh(V, F), show_edges=True)
    p.add_mesh(arrows)
    p.show()

def plot_lines(p, v0, v1, color):
    tmp = np.zeros((v0.shape[0] * 2, 3))
    tmp[0::2] = v0
    tmp[1::2] = v1
    p.add_lines(tmp, color=color)

In [3]:
v, f = igl.read_triangle_mesh("data/bunny.off")
to_pyvista_mesh(v, f).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:63755/index.html?ui=P_0x29d1b5ae8d0_0&reconnect=auto" class="pyvis…

# Vertex normal

In [4]:
#Standard face normal
plot_mesh_with_normals(v, f, igl.per_vertex_normals(v, f), factor=0.01)

Widget(value='<iframe src="http://localhost:63755/index.html?ui=P_0x29d1b751950_1&reconnect=auto" class="pyvis…

In [5]:
#Area-weighted face normal
plot_mesh_with_normals(v, f, igl.per_vertex_normals(v, f, igl.PER_VERTEX_NORMALS_WEIGHTING_TYPE_AREA), factor=0.01)

Widget(value='<iframe src="http://localhost:63755/index.html?ui=P_0x29d20ad9810_2&reconnect=auto" class="pyvis…

In [6]:
#Mean-curvature normal
def mean_curvature_threshold_normals(V, F, eps=1e-3):
    L = igl.cotmatrix(V, F)

    HN = -L @ V
    HN_mag = np.linalg.norm(HN, axis=1)

    N_area = igl.per_vertex_normals(V, F, igl.PER_VERTEX_NORMALS_WEIGHTING_TYPE_AREA)
    area_mag = np.linalg.norm(N_area, axis=1)
    N_area_unit = np.zeros_like(N_area)
    valid_area = area_mag > 0
    N_area_unit[valid_area] = N_area[valid_area] / area_mag[valid_area, None]

    # Use mean-curvature direction where strong enough, area-weighted elsewhere
    N = np.zeros_like(V)
    use_hn = HN_mag > eps
    N[use_hn] = HN[use_hn] / HN_mag[use_hn, None]
    N[~use_hn] = N_area_unit[~use_hn]

    # Optional orientation consistency with area normals
    flip = np.einsum("ij,ij->i", N, N_area_unit) < 0
    N[flip] *= -1.0

    # Final normalization
    Nmag = np.linalg.norm(N, axis=1)
    good = Nmag > 0
    N[good] /= Nmag[good, None]

    return N, HN_mag

eps = 1e-3
N_mc, HN_mag = mean_curvature_threshold_normals(v, f, eps=eps)

plot_mesh_with_normals(v, f, N_mc, factor=0.01)


Widget(value='<iframe src="http://localhost:63755/index.html?ui=P_0x29d20b67310_3&reconnect=auto" class="pyvis…

In [7]:
#PCA normal
def pca_vertex_normals(V, F, k):
    V = np.asarray(V, dtype=np.float64)
    n = V.shape[0] 

    N = np.zeros((n, 3), dtype=np.float64)

    for i in range(n):
        d2 = np.sum((V - V[i]) ** 2, axis=1)
        nn = np.argpartition(d2, k + 1)[: k + 1]   # includes self
        nn = nn[nn != i]                            # remove self
        if nn.shape[0] > k:
            nn = nn[:k]

        P = V[nn]
        P = np.vstack([V[i], P])                   

        C = np.cov(P, rowvar=False, bias=True)

        # Smallest eigenvalue = plane normal
        w, U = np.linalg.eigh(C)                    
        N[i] = U[:, 0]

    # Orient consistently with area-weighted normals
    N_area = igl.per_vertex_normals(V, F, igl.PER_VERTEX_NORMALS_WEIGHTING_TYPE_AREA)
    flip = np.einsum("ij,ij->i", N, N_area) < 0
    N[flip] *= -1.0

    # Normalize
    mag = np.linalg.norm(N, axis=1)
    good = mag > 0
    N[good] /= mag[good, None]

    return N

k = 30
N_pca = pca_vertex_normals(v, f, k)
plot_mesh_with_normals(v, f, N_pca, factor=0.01)

Widget(value='<iframe src="http://localhost:63755/index.html?ui=P_0x29d36b8ab50_4&reconnect=auto" class="pyvis…

In [8]:
#Quadratic fitting normal
def quadratic_fitting_normals(V, F, k):
    V = np.asarray(V, dtype=np.float64)
    n = V.shape[0]

    N = np.zeros((n, 3), dtype=np.float64)

    for i in range(n):
        # Same as PCA
        d2 = np.sum((V - V[i]) ** 2, axis=1)
        nn = np.argpartition(d2, k + 1)[:k + 1]
        nn = nn[nn != i]
        if nn.shape[0] > k:
            nn = nn[:k]

        P = np.vstack([V[i], V[nn]])

        # PCA local frame
        C = np.cov(P, rowvar=False, bias=True)
        w, U = np.linalg.eigh(C)  
       
        # Project into local frame centered at vertex i
        dP = P - V[i]
        u_coords = dP @ U[:, 1]
        v_coords = dP @ U[:, 2]
        h_coords = dP @ U[:, 0]

        # Fit quadratic: a u^2 + b uv + c v^2 + d u + e v + const
        A = np.column_stack([u_coords**2, u_coords * v_coords, v_coords**2, u_coords, v_coords, np.ones(len(u_coords))])
        coeffs, _, _, _ = np.linalg.lstsq(A, h_coords, rcond=None)

        fu = coeffs[3] # At u=v=0: f_u = d = coeffs[3], f_v = e = coeffs[4]
        fv = coeffs[4]

        Su = U[:, 1] + fu * U[:, 0] # S_u = e1 + fu*e0
        Sv = U[:, 2] + fv * U[:, 0] # S_v = e2 + fv*e0

        normal = np.cross(Su, Sv)
        mag = np.linalg.norm(normal)
        if mag > 0:
            normal /= mag
        N[i] = normal

    N_area = igl.per_vertex_normals(V, F, igl.PER_VERTEX_NORMALS_WEIGHTING_TYPE_AREA)
    flip = np.einsum("ij,ij->i", N, N_area) < 0
    N[flip] *= -1.0

    mag = np.linalg.norm(N, axis=1)
    good = mag > 0
    N[good] /= mag[good, None]

    return N

k = 30
N_qf = quadratic_fitting_normals(v, f, k)
plot_mesh_with_normals(v, f, N_qf, factor=0.01)

Widget(value='<iframe src="http://localhost:63755/index.html?ui=P_0x29d36c19750_5&reconnect=auto" class="pyvis…

# Curvature

In [ ]:
#gaussian curvature
k = igl.gaussian_curvature(v, f)
mesh = to_pyvista_mesh(v, f)
mesh.point_data["gaussian_curvature"] = k

p = pv.Plotter()
p.add_mesh(mesh, scalars="gaussian_curvature", cmap="coolwarm", show_edges=False)
p.show()

In [17]:
# principal curvature
pd1, pd2, pv1, pv2, pv3 = igl.principal_curvature(v, f)

start_k_min = v
end_k_min   = v + 0.005 * pd2   # min principal direction 

start_k_max = v
end_k_max   = v + 0.005 * pd1   # max principal direction

mesh = to_pyvista_mesh(v, f)
mesh.point_data["mean_curvature"] = (pv1 + pv2) / 2

p = pv.Plotter()
p.add_mesh(mesh, show_edges=True, scalars="mean_curvature", cmap="coolwarm")
plot_lines(p, start_k_min, end_k_min, "red")
plot_lines(p, start_k_max, end_k_max, "green")
p.show()

Widget(value='<iframe src="http://localhost:63755/index.html?ui=P_0x29d485529d0_14&reconnect=auto" class="pyvi…

# Smoothing with the Laplacian

In [11]:
from scipy.sparse.linalg import spsolve
import scipy.sparse as sp

In [ ]:
# Explicit laplacian
v, f = igl.read_triangle_mesh("data/cow.off")
l = igl.cotmatrix(v, f)
m = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_BARYCENTRIC)
Minv = sp.diags(1 / m.diagonal())

n = igl.per_vertex_normals(v, f) * 0.5 + 0.5
c = np.linalg.norm(n, axis=1)

vs = [v]
cs = [c]

lam = 0.000001
num_iters = 1000

for i in range(num_iters):
    # Explicit update: V_new = V + lambda * M^{-1} L V
    v = v + lam * (Minv @ (l @ v))
    n = igl.per_vertex_normals(v, f) * 0.5 + 0.5
    c = np.linalg.norm(n, axis=1)
    vs.append(v.copy())
    cs.append(c)

p = pv.Plotter(shape=(1, 2))
for col, (idx, title) in enumerate([(0, "Original"), (-1, f"Smoothed x{num_iters}")]):
    mesh = to_pyvista_mesh(vs[idx], f)
    mesh.point_data["color"] = cs[idx]
    p.subplot(0, col)
    p.add_text(title)
    p.add_mesh(mesh, scalars="color", cmap="viridis", show_edges=False)
p.show()

In [ ]:
# Implicit laplacian
# taken from https://libigl.github.io/libigl-python-bindings/tut-chapter1/#laplacian
v, f = igl.read_triangle_mesh("data/cow.off")
l = igl.cotmatrix(v, f)

n = igl.per_vertex_normals(v, f)*0.5+0.5
c = np.linalg.norm(n, axis=1)


vs = [v]
cs = [c]
for i in range(1):
    # Implicit update: (M - lambda * L) V_new = M V
    m = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_BARYCENTRIC)
    s = (m - 0.001 * l)
    b = m.dot(v)
    v = spsolve(s, m.dot(v))
    n = igl.per_vertex_normals(v, f)*0.5+0.5
    c = np.linalg.norm(n, axis=1)
    vs.append(v)
    cs.append(c)

p = pv.Plotter(shape=(1, 2))
for col, (idx, title) in enumerate([(0, "Original"), (-1, "Smoothed x1")]):
    mesh = to_pyvista_mesh(vs[idx], f)
    mesh.point_data["color"] = cs[idx]
    p.subplot(0, col)
    p.add_text(title)
    p.add_mesh(mesh, scalars="color", cmap="viridis", show_edges=False)
p.show()

Widget(value='<iframe src="http://localhost:59150/index.html?ui=P_0x1eceffb38d0_34&reconnect=auto" class="pyvi…